In [1]:
import sys
import os

import torch
from dataclasses import dataclass, field
from abc import ABC, abstractmethod
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import gym
from typing import Any, NamedTuple

sys.path.append(os.path.expanduser("~")+"/dev/MLStudy/mllib")

import rllib

In [2]:
import gym_simpletetris
class ClipRewardEnv(gym.RewardWrapper):
    """
    Clips the reward to {+1, 0, -1} by its sign.
    Args:
        env (gym.Env): The environment to wrap
    """

    def __init__(self, env: gym.Env):
        gym.RewardWrapper.__init__(self, env)
    
    def reward(self, reward: float) -> float:
        return np.sign(reward)

np.float = np.float32
envname = 'BreakoutNoFrameskip-v4'
def create_env(mode='rgb_array'):
    env =  gym.wrappers.AtariPreprocessing(
        gym.make(envname, render_mode='rgb_array'),
        noop_max=30,                   # 30 random actions a the beginning of an episode
        frame_skip=4,
        screen_size=84,                # Changes observation size to 84x84
        terminal_on_life_loss=True,    # Returns done=True if episode terminates
        grayscale_obs=True,            # Convert RGB to grayscale
        scale_obs=True,                # Scales observations to range 0-1
    )
    env = ClipRewardEnv(env)                           # Clip all rewards to {-1, 0, +1}
    env = gym.wrappers.FrameStack(env, 4) 
    env = rllib.TorchObservationWrapper(env)
    return env

In [3]:
env = create_env()

A.L.E: Arcade Learning Environment (version 0.8.1+53f58b7)
[Powered by Stella]


In [4]:
N = 64

def init_layer(layer, std, bias):
    torch.nn.init.orthogonal_(layer.weight, std)
    torch.nn.init.constant_(layer.bias, bias)
    return layer

class NeuralNetwork(nn.Module):
    def __init__(self, obs_size, act_size):
        super().__init__()
        self.hidden = nn.Sequential(init_layer(nn.Conv2d(4, 32, kernel_size=8, stride=4), np.sqrt(2), 0.0),
            nn.ReLU(),
            init_layer(nn.Conv2d(32, 64, kernel_size=4, stride=2), np.sqrt(2), 0.0),
            nn.ReLU(),
            init_layer(nn.Conv2d(64, 64, kernel_size=3, stride=1), np.sqrt(2), 0.0),
            nn.ReLU(),
            nn.Flatten(),
            init_layer(nn.Linear(64*7*7,N), np.sqrt(2), 0),
            nn.ReLU()
        )
        self.hidden2 = nn.Sequential(init_layer(nn.Conv2d(4, 32, kernel_size=8, stride=4), np.sqrt(2), 0.0),
            nn.ReLU(),
            init_layer(nn.Conv2d(32, 64, kernel_size=4, stride=2), np.sqrt(2), 0.0),
            nn.ReLU(),
            init_layer(nn.Conv2d(64, 64, kernel_size=3, stride=1), np.sqrt(2), 0.0),
            nn.ReLU(),
            nn.Flatten(),
            init_layer(nn.Linear(64*7*7,N), np.sqrt(2), 0),
            nn.ReLU()
        )
        self.action_head = init_layer(nn.Linear(N, act_size), 0.01, 0)
        self.value_head = init_layer(nn.Linear(N, 1), 1, 0)
    
    def get_policy(self, x):
        if x.ndim == 3:
            x = torch.reshape(x, (-1,) + x.shape)
        logits = self.action_head(self.hidden(x))
        return torch.distributions.Categorical(logits=logits)
    
    def get_value(self, x):
        if x.ndim == 3:
            x = torch.reshape(x, (-1,) + x.shape)
        return self.value_head(self.hidden2(x))


In [5]:
n = env.observation_space.shape[0]*env.observation_space.shape[1]
cuda = torch.device('cuda')
model = NeuralNetwork(n, env.action_space.n)
model.load_state_dict(torch.load("breakout3.model"))
agent = rllib.A2CMLAgent(model, cuda)

In [6]:
rllib.run_jupyter(create_env, agent, cuda, 602400)

KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), "breakout4.model")

In [7]:

import torch
from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter()

model_sizes = [32,48,64,80,96]
eps = [0.1,0.2,0.3,0.4]
bs = [16,32,48,128]
gs = [0.9,0.95, 0.99, 0.995]
lrs = [1e-5, 1e-4, 1e-3, 1e-2]
res = []
# random_seed =532421
# torch.manual_seed(random_seed)
# torch.cuda.manual_seed(random_seed)
# torch.backends.cudnn.deterministic = True
# torch.backends.cudnn.benchmark = False
# np.random.seed(random_seed)
n = env.observation_space.shape[0]*env.observation_space.shape[1]
#model = NeuralNetwork(n, env.action_space.n).cuda()
agent = rllib.A2CMLAgent(model, cuda)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
options = rllib.PPOOptions(optimizer)
options.gamma = 0.99
options.epsilon = 0.1
options.batch_nums = 4
options.train_count = 4
options.epochs = 10
options.entropy = 0.00
options.num_envs = 8
options.max_episode_len = 1024
ri = 0
li = 0
def add_reward(x):
    global ri
    writer.add_scalar("Reward/train", x, ri)
    ri += 1
    
def add_loss(x):
    global li
    for k,v in x.items():
        writer.add_scalar("Loss/" + k, v, li)
    li += 1

options.report_reward = add_reward
options.report_train = add_loss
import cProfile
stats = rllib.ppo_train(create_env, agent, cuda, options)
res.append((stats,agent))

In [ ]:
cuda = torch.device('cuda')     # Default CUDA device


In [12]:
cuda

device(type='cuda')